In [ ]:
# H1 CELL 1 — era-split exposure + structural change regression
!pip -q install linearmodels 2>/dev/null
import pandas as pd, numpy as np, os, statsmodels.api as sm
from google.colab import drive; drive.mount('/content/drive')

BASE = '/content/drive/My Drive/Data/Elevar'
KEYS = ['Dist Code','Year','State Code','State Name','Dist Name']
E1, E2 = (1966,1990), (1991,2015)          # eras

def rd(f):
    p=os.path.join(BASE,f)
    try: d=pd.read_csv(p, low_memory=False)
    except UnicodeDecodeError: d=pd.read_csv(p, encoding='latin-1', low_memory=False)
    if 'Year' in d.columns:
        d['Year']=pd.to_numeric(d['Year'].astype(str).str.extract(r'(\d{4})')[0], errors='coerce')
    v=[c for c in d.columns if c not in KEYS]
    d[v]=d[v].apply(pd.to_numeric, errors='coerce').mask(lambda x: x<0)
    return d[d['Dist Code']<800]

# ---------- panel ----------
r     = pd.read_parquet(f'{BASE}/rainfall_shock.parquet'); r=r[r['Dist Code']<800]
crops = pd.read_parquet(f'{BASE}/crops_clean.parquet')
P = crops.merge(r, on=['Dist Code','Year'], how='inner').dropna(subset=['ln_ag_output','rain_z'])
P['z_neg']=np.minimum(P.rain_z,0); P['z_pos']=np.maximum(P.rain_z,0)

def era_slopes(lo, hi, tag):
    rows=[]
    for dc,d in P[P.Year.between(lo,hi)].groupby('Dist Code'):
        d=d.sort_values('Year')
        if len(d)<15 or (d.z_neg<0).sum()<6: continue
        t=d.Year.values-d.Year.values.mean()
        f=sm.OLS(d.ln_ag_output.values, sm.add_constant(np.column_stack([d.z_neg,d.z_pos,t]))).fit()
        rows.append({'Dist Code':dc,f'b_{tag}':f.params[1],f'se_{tag}':f.bse[1],f'n_{tag}':len(d)})
    A=pd.DataFrame(rows)
    mu=A[f'b_{tag}'].mean(); tau2=max(0.,A[f'b_{tag}'].var(ddof=1)-(A[f'se_{tag}']**2).mean())
    w=tau2/(tau2+A[f'se_{tag}']**2)
    A[f'exp_{tag}']=mu+w*(A[f'b_{tag}']-mu); A[f'w_{tag}']=w
    print(f'{tag}: {len(A)} districts | mean {mu:+.3f} | tau2 {tau2:.4f} | shrink w {w.mean():.2f}')
    return A

A1=era_slopes(*E1,'e1'); A2=era_slopes(*E2,'e2')
D=A1.merge(A2,on='Dist Code',how='inner')
D['d_exposure']=D.exp_e2-D.exp_e1
D['w_pair']=1/(1/D.w_e1.clip(.01)+1/D.w_e2.clip(.01))       # joint precision weight
meta=crops.groupby('Dist Code')[['Dist Name','State Name']].first()
D=D.join(meta,on='Dist Code')
print(f'\nBOTH ERAS: {len(D)} districts')
print(f'  era1 mean exposure {D.exp_e1.mean():+.3f} | era2 {D.exp_e2.mean():+.3f} | change {D.d_exposure.mean():+.3f}')
tt=sm.stats.DescrStatsW(D.d_exposure, weights=D.w_pair).ttest_mean(0)
print(f'  weighted mean change t={tt[0]:.2f}, p={tt[1]:.3f}')
print(f'  districts that became LESS exposed: {(D.d_exposure<0).sum()} of {len(D)}')
print(f'  corr(era1 exposure, change) = {D.exp_e1.corr(D.d_exposure):+.3f}  <- negative = convergence')

print('\nBIGGEST IMPROVERS:'); print(D.nsmallest(10,'d_exposure')[['Dist Name','State Name','exp_e1','exp_e2','d_exposure']].round(3).to_string(index=False))
print('\nBIGGEST DETERIORATIONS:'); print(D.nlargest(10,'d_exposure')[['Dist Name','State Name','exp_e1','exp_e2','d_exposure']].round(3).to_string(index=False))

# ---------- era-mean structural features ----------
def era_mean(df, cols, lo, hi):
    w=df[df.Year.between(lo,hi)]
    return w.groupby('Dist Code')[cols].mean()

def build(lo,hi):
    F=pd.DataFrame(index=pd.Index(sorted(P['Dist Code'].unique()),name='Dist Code'))
    lu=rd('icrisat_landuse.csv')
    L=era_mean(lu,['TOTAL AREA (1000 ha)','NET CROPPED AREA (1000 ha)','CROPING INTENSITY (Percent)'],lo,hi)
    ir=rd('icrisat_irr_sourcewise.csv')
    I=era_mean(ir,['CANALS AREA (1000 ha)','TANKS AREA (1000 ha)','TUBE WELLS AREA (1000 ha)','NET AREA (1000 ha)'],lo,hi)
    den=I['NET AREA (1000 ha)'].replace(0,np.nan)
    F['irr_share']      = I['NET AREA (1000 ha)']/L['NET CROPPED AREA (1000 ha)']
    F['canal_share']    = I['CANALS AREA (1000 ha)']/den
    F['tubewell_share'] = I['TUBE WELLS AREA (1000 ha)']/den
    F['crop_intensity'] = L['CROPING INTENSITY (Percent)']
    C=era_mean(crops,['msp_cereal_share','coarse_cereal_share','oilseeds_share','cash_crop_share','crop_hhi'],lo,hi)
    F=F.join(C)
    fz=rd('icrisat_fertilizer.csv'); F=F.join(era_mean(fz,['TOTAL PER HA OF GCA (Kg per ha)'],lo,hi).rename(columns={'TOTAL PER HA OF GCA (Kg per ha)':'fert_per_ha'}))
    pop=rd('icrisat_census_population.csv')
    PP=era_mean(pop,['TOTAL POPULATION (1000 Number)','NON AGRICULTURAL WORKERS POPULATION (1000 Number)','TOTAL WORKERS POPULATION (1000 Number)'],lo,hi)
    F['nonfarm_worker_share']=PP['NON AGRICULTURAL WORKERS POPULATION (1000 Number)']/PP['TOTAL WORKERS POPULATION (1000 Number)']
    F['pop_density']=PP['TOTAL POPULATION (1000 Number)']*100/L['TOTAL AREA (1000 ha)']
    bk=rd('icrisat_banks.csv'); B=era_mean(bk,['BANKS NUMBER (Number)'],lo,hi)
    F['banks_per_lakh']=B['BANKS NUMBER (Number)']*100/PP['TOTAL POPULATION (1000 Number)']
    return F

F1,F2 = build(*E1), build(*E2)
VARS=['irr_share','canal_share','tubewell_share','crop_intensity','msp_cereal_share',
      'coarse_cereal_share','oilseeds_share','cash_crop_share','crop_hhi','fert_per_ha',
      'nonfarm_worker_share','pop_density','banks_per_lakh']
Δ=(F2[VARS]-F1[VARS]).add_prefix('d_')
print('\nSTRUCTURAL CHANGE, era1 -> era2 (mean):')
print(pd.DataFrame({'era1':F1[VARS].mean(),'era2':F2[VARS].mean(),'change':(F2[VARS]-F1[VARS]).mean()}).round(3).to_string())

R = D.set_index('Dist Code').join(Δ).join(F1[VARS].add_prefix('base_'))
R = R.dropna(subset=['d_exposure'])
print(f'\nregression frame: {R.shape}')

# ---------- the regression: does structural change explain exposure change? ----------
X = R[[f'd_{v}' for v in VARS]+['base_irr_share','base_nonfarm_worker_share','exp_e1']].copy()
X = X.fillna(X.median())
X = (X-X.mean())/X.std()                       # standardise -> coefficients comparable
m = sm.WLS(R.d_exposure.values, sm.add_constant(X), weights=R.w_pair.values).fit(cov_type='HC3')
print('\nΔEXPOSURE ~ ΔSTRUCTURE  (standardised, WLS by joint precision, HC3)')
print(f'R2 {m.rsquared:.3f} | N {int(m.nobs)}\n')
out=pd.DataFrame({'coef':m.params,'se':m.bse,'t':m.tvalues,'p':m.pvalues}).drop('const')
print(out.reindex(out.t.abs().sort_values(ascending=False).index).round(3).to_string())
print('\nnegative coef = that increase REDUCED exposure (made the district more resilient)')

D.to_parquet(f'{BASE}/h1_era_exposure.parquet', index=False)
R.reset_index().to_parquet(f'{BASE}/h1_regression_frame.parquet', index=False)
print('\nsaved h1_era_exposure.parquet, h1_regression_frame.parquet')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.7 MB/s eta 0:00:00
Mounted at /content/drive
e1: 249 districts | mean +0.273 | tau2 0.1412 | shrink w 0.89
e2: 273 districts | mean +0.215 | tau2 0.0863 | shrink w 0.88

BOTH ERAS: 248 districts
  era1 mean exposure +0.253 | era2 +0.196 | change -0.057
  weighted mean change t=-2.72, p=0.008
  districts that became LESS exposed: 157 of 248
  corr(era1 exposure, change) = -0.578  <- negative = convergence

BIGGEST IMPROVERS:
    Dist Name       State Name  exp_e1  exp_e2  d_exposure
    Jaisalmer        Rajasthan   1.306   0.493      -0.814
      Jodhpur        Rajasthan   1.259   0.616      -0.643
     Bhandara      Maharashtra   0.785   0.143      -0.642
  Banaskantha          Gujarat   0.857   0.266      -0.591
      Kinnaur Himachal Pradesh   0.316  -0.240      -0.556
        Kutch          Gujarat   0.872   0.332      -0.541
      Mirzpur    Uttar Pradesh   0.485  -0.028      -0.513
       Barmer        Rajasthan   0.934   0

In [ ]:
# H1 CELL 2 — is nonfarm_worker_share broken?
pop = rd('icrisat_census_population.csv')
C = ['TOTAL CULTIVATORS POPULATION (1000 Number)',
     'TOTAL AGRICULTURAL LABOUR POPULATION (1000 Number)',
     'AGRICULTURAL WORKERS POPULATION (1000 Number)',
     'NON AGRICULTURAL WORKERS POPULATION (1000 Number)',
     'TOTAL WORKERS POPULATION (1000 Number)',
     'TOTAL POPULATION (1000 Number)']
d = pop.dropna(subset=C).copy()
d['cult_plus_lab'] = d[C[0]]+d[C[1]]
d['ag_plus_nonag'] = d[C[2]]+d[C[3]]
print('does AG WORKERS == CULTIVATORS + AG LABOUR?')
print((d.cult_plus_lab/d[C[2]]).describe().round(3).to_string())
print('\ndoes AG + NONAG == TOTAL WORKERS?')
print((d.ag_plus_nonag/d[C[4]]).describe().round(3).to_string())
print('\nWFPR (total workers / population):', round((d[C[4]]/d[C[5]]).mean(),3), ' <- should be ~0.35-0.42')
print('\nshares of TOTAL WORKERS, by census year:')
print(d.assign(cult=d[C[0]]/d[C[4]], aglab=d[C[1]]/d[C[4]],
               agw=d[C[2]]/d[C[4]], nonag=d[C[3]]/d[C[4]])
       .groupby('Year')[['cult','aglab','agw','nonag']].mean().round(3).to_string())

does AG WORKERS == CULTIVATORS + AG LABOUR?
count    1653.000
mean        1.000
std         0.000
min         0.999
25%         1.000
50%         1.000
75%         1.000
max         1.000

does AG + NONAG == TOTAL WORKERS?
count    1654.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0

WFPR (total workers / population): 0.892  <- should be ~0.35-0.42

shares of TOTAL WORKERS, by census year:
       cult  aglab    agw  nonag
Year                            
1961  0.294  0.087  0.380  0.620
1971  0.182  0.085  0.268  0.732
1981  0.185  0.087  0.272  0.728
1991  0.175  0.091  0.266  0.734
2001  0.182  0.117  0.299  0.701
2011  0.153  0.142  0.295  0.705


In [ ]:
# H1 CELL 3 — rebuild nonfarm share on a defensible denominator
pop = rd('icrisat_census_population.csv')
C_cult, C_lab = 'TOTAL CULTIVATORS POPULATION (1000 Number)','TOTAL AGRICULTURAL LABOUR POPULATION (1000 Number)'
C_tot = 'TOTAL POPULATION (1000 Number)'
d = pop.dropna(subset=[C_cult,C_lab,C_tot]).copy()
d['agworkers_pc'] = (d[C_cult]+d[C_lab])/d[C_tot]      # ag workers per head of population
print('ag workers per capita, by census year (expect ~0.25-0.35, falling):')
print(d.groupby('Year').agworkers_pc.mean().round(3).to_string())
print('\ncross-district spread, 2011:')
print(d[d.Year==2011].agworkers_pc.describe().round(3).to_string())
print('\nlowest (most non-farm) 2011:')
print(d[d.Year==2011].nsmallest(8,'agworkers_pc')[['Dist Name','State Name','agworkers_pc']].round(3).to_string(index=False))
print('\nhighest (most agrarian) 2011:')
print(d[d.Year==2011].nlargest(8,'agworkers_pc')[['Dist Name','State Name','agworkers_pc']].round(3).to_string(index=False))

ag workers per capita, by census year (expect ~0.25-0.35, falling):
Year
1961    0.346
1971    0.249
1981    0.243
1991    0.241
2001    0.258
2011    0.249

cross-district spread, 2011:
count    279.000
mean       0.249
std        0.098
min        0.004
25%        0.186
50%        0.260
75%        0.320
max        0.478

lowest (most non-farm) 2011:
         Dist Name  State Name  agworkers_pc
            Bombay Maharashtra         0.004
          Thrissur      Kerala         0.037
Thiruvananthapuram      Kerala         0.040
        Malappuram      Kerala         0.043
         Alappuzha      Kerala         0.046
       Kanyakumari  Tamil Nadu         0.047
        Eranakulam      Kerala         0.049
         Kozhikode      Kerala         0.053

highest (most agrarian) 2011:
Dist Name       State Name  agworkers_pc
    Kullu Himachal Pradesh         0.478
    Dangs          Gujarat         0.460
   Mandla   Madhya Pradesh         0.460
   Jhabua   Madhya Pradesh         0.442
  Kinn

In [ ]:
# H1 CELL 4 — rerun H1 and the SHAP model with the corrected variable
import pandas as pd, numpy as np, os, shap, statsmodels.api as sm
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer

pop = rd('icrisat_census_population.csv')
C_cult,C_lab,C_tot = ('TOTAL CULTIVATORS POPULATION (1000 Number)',
                      'TOTAL AGRICULTURAL LABOUR POPULATION (1000 Number)',
                      'TOTAL POPULATION (1000 Number)')
pop = pop.dropna(subset=[C_cult,C_lab,C_tot]).copy()
pop['agworkers_pc'] = (pop[C_cult]+pop[C_lab])/pop[C_tot]

# ---------- (a) H1 rerun ----------
def agpc(lo,hi):
    return pop[pop.Year.between(lo,hi)].groupby('Dist Code').agworkers_pc.mean()
a1,a2 = agpc(1961,1990), agpc(1991,2011)

D = pd.read_parquet(f'{BASE}/h1_era_exposure.parquet').set_index('Dist Code')
R = pd.read_parquet(f'{BASE}/h1_regression_frame.parquet').set_index('Dist Code')
R['d_agworkers_pc']    = (a2-a1)
R['base_agworkers_pc'] = a1
VARS2 = ['d_irr_share','d_canal_share','d_tubewell_share','d_crop_intensity','d_msp_cereal_share',
         'd_coarse_cereal_share','d_oilseeds_share','d_cash_crop_share','d_crop_hhi','d_fert_per_ha',
         'd_agworkers_pc','d_pop_density','d_banks_per_lakh',
         'base_irr_share','base_agworkers_pc','exp_e1']
X = R[VARS2].copy(); X = X.fillna(X.median()); X=(X-X.mean())/X.std()
m = sm.WLS(R.d_exposure.values, sm.add_constant(X), weights=R.w_pair.values).fit(cov_type='HC3')
print(f'H1 RERUN — R2 {m.rsquared:.3f}, N {int(m.nobs)}')
o = pd.DataFrame({'coef':m.params,'se':m.bse,'t':m.tvalues,'p':m.pvalues}).drop('const')
print(o.reindex(o.t.abs().sort_values(ascending=False).index).round(3).to_string())
print('note: d_agworkers_pc POSITIVE = rising farm dependence raised exposure\n')

# ---------- (b) SHAP model rerun ----------
F = pd.read_parquet(f'{BASE}/features.parquet').set_index('Dist Code')
F['agworkers_pc'] = pop[pop.Year>=2001].groupby('Dist Code').agworkers_pc.mean()
d = F.dropna(subset=['exposure']).copy()
w = d['w'].astype(float).values if 'w' in d else np.ones(len(d))
y = d['exposure'].astype(float).values
FEATS = ['coarse_cereal_share','oilseeds_share','cash_crop_share','irr_share','canal_share',
         'tubewell_share','kharif_normal','rain_cv','crop_intensity','marginal_share',
         'agworkers_pc','pop_density','banks_per_lakh','milk_kg_pc']
X2 = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(d[FEATS].astype(float)),
                  columns=FEATS, index=d.index)
cv = KFold(5, shuffle=True, random_state=7)
g = HistGradientBoostingRegressor(max_depth=3,max_iter=400,learning_rate=0.05,
                                  min_samples_leaf=15,l2_regularization=1.0,random_state=7)
oof = cross_val_predict(g, X2, y, cv=cv, params={'sample_weight':w})
print(f'SHAP MODEL RERUN — oof R2 {1-((y-oof)**2).sum()/((y-y.mean())**2).sum():.3f} (was 0.498 decomp)')
g.fit(X2,y,sample_weight=w)
S = pd.DataFrame(shap.TreeExplainer(g).shap_values(X2), columns=FEATS, index=d.index)
print('\nmean |SHAP|:'); print(S.abs().mean().sort_values(ascending=False).round(4).to_string())
print(f"\ncorr(agworkers_pc, exposure) = {d.agworkers_pc.corr(d.exposure):+.3f}  (old nonfarm var was -0.255)")
print('districts whose #1 driver is each feature:')
print(S.abs().idxmax(axis=1).value_counts().head(6).to_string())

S.join(d[['Dist Name','State Name']]).reset_index().to_parquet(f'{BASE}/shap_decomp.parquet', index=False)
F.reset_index().to_parquet(f'{BASE}/features.parquet', index=False)
print('\nsaved (overwrote shap_decomp.parquet, features.parquet)')

H1 RERUN — R2 0.512, N 248
                        coef     se      t      p
exp_e1                -0.134  0.019 -7.123  0.000
d_irr_share           -0.035  0.012 -2.921  0.003
d_oilseeds_share       0.042  0.018  2.329  0.020
d_pop_density         -0.025  0.011 -2.257  0.024
d_msp_cereal_share    -0.033  0.018 -1.846  0.065
d_crop_intensity       0.020  0.011  1.776  0.076
base_irr_share        -0.021  0.014 -1.472  0.141
d_cash_crop_share      0.019  0.013  1.436  0.151
d_agworkers_pc         0.016  0.012  1.379  0.168
d_banks_per_lakh       0.012  0.011  1.141  0.254
d_crop_hhi             0.019  0.016  1.138  0.255
base_agworkers_pc     -0.006  0.013 -0.481  0.631
d_fert_per_ha          0.005  0.011  0.439  0.661
d_coarse_cereal_share -0.005  0.015 -0.355  0.722
d_canal_share          0.003  0.010  0.295  0.768
d_tubewell_share      -0.001  0.010 -0.065  0.948
note: d_agworkers_pc POSITIVE = rising farm dependence raised exposure

SHAP MODEL RERUN — oof R2 0.511 (was 0.498 decomp)
